# modular-diffusion: the joint pass, how parts are born, members on averaged weights

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/modular-diffusion/blob/v0.6.0/colab/parts.ipynb)

A model built on a frozen base (blocks trained on a finished model that no longer moves) stays behind the same model trained whole. This notebook measures what a joint pass buys back: training on from such a build with only its top part, its top part and one or two blocks of the base, or everything set trainable (`geolip.modular_diffusion.experiments.stages`, variant `thaw`), against the reference trained straight through at the same compute. It also measures how a content expert is best born (silent, or with its head solved in closed form before any training), whether content experts are better kept per class or per class and stage, whether a class a model never saw can be added by a solve alone (`experiments.content`, `experiments.closed_forms`), and what frequency-band members add to a finished model.

1. install at a release tag and print the environment;
2. the data with its content hash, the finished runs this batch starts from (fetched from the Hub), a smoke run of the picture score and of the new paths, and the size of the job list;
3. the job list; each finished job is shipped to the Hub at once; a progress line a minute; the summary; then the machine is given back.

A stopped session resumes by running the cells again: finished jobs keep their markers under `OUT`, and on a new machine the jobs already on the Hub are fetched back first.

In [ ]:
# C1 - install at a tag (extras on the name); torch is not a dependency, so the runtime's torch stays
%pip install -q "modular-diffusion[data,hub,judge,guard] @ git+https://github.com/AbstractEyes/modular-diffusion@v0.6.0"
import os, sys, time, json, shutil, threading, subprocess
import torch, geolip.modular_diffusion as MD
from geolip.modular_diffusion import environment
assert tuple(int(x) for x in MD.__version__.split(".")) >= (0, 6, 0), "RESTART REQUIRED: Runtime > Restart session, then run C1 again"
environment.check(None)
print("card:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (choose a GPU runtime)", "| cpus:", os.cpu_count())
OUT = "/content/parts"               # the output root; keep it the same to resume
os.makedirs(OUT, exist_ok=True)

The data comes from the Hub as parquet files and becomes one npz of uint8 arrays with a content hash. The finished runs this batch starts from are fetched from the results repository: the reference model, the model that never reads the class and the model trained without one class (seeds 0 and 1, each with its averaged weights), and the two builds on a frozen 4-block model. The smoke run trains the reference for 60 steps and scores it (the score is checked against a second implementation and the class reader on the test pictures), then runs every new path for a few steps.

In [ ]:
# C2 - the data, the runs this batch starts from, a smoke run, the job list's size
from huggingface_hub import snapshot_download
from geolip.modular_diffusion import data, score, linker
from geolip.modular_diffusion.beds import PicturesBed
from geolip.modular_diffusion.experiments import time_split, content, closed_forms, stages
NPZ = f"{OUT}/data/cifar10_32.npz"
REF = f"{OUT}/data/reference_inception.npz"     # the reference statistics: computed once here, read by every job
if not os.path.exists(NPZ):
    snapshot_download(repo_id="uoft-cs/cifar10", repo_type="dataset", allow_patterns=["plain_text/*.parquet"], local_dir=f"{OUT}/parquet")
    data.cifar_npz(parquet_dir=f"{OUT}/parquet/plain_text", out=NPZ)
EXPECTED_CONTENT_HASH = "be50ca799cbe6795ff13bca750f53f16c974b9f4903082fc06031b9719700016"
got = data.npz_content_hash(NPZ)
print("content hash", got, "| expected", EXPECTED_CONTENT_HASH or "(not filled)")
assert not EXPECTED_CONTENT_HASH or got == EXPECTED_CONTENT_HASH, "the npz is not the expected data"

REPO = "AbstractPhil/geolip-multirotary-diffusion-collective"   # the results repository ("" in C3 = nothing shipped)
BODIES = [f"{v}_s{s}" for v in ("plain", "blind", "blind9") for s in (0, 1)]   # finished runs with averaged weights
BUILDS = [f"{v}_s{s}" for v in ("tails_k1", "grow_1x3") for s in (0, 1)]       # builds on a frozen 4-block model
STORED = "blind_experts_all_avg_s0"                                            # a finished job that is run again here
hub_dir = snapshot_download(repo_id=REPO, allow_patterns=[f"modular_diffusion/content/{t}/*" for t in BODIES]
                            + [f"modular_diffusion/stages/{t}/*" for t in BUILDS]
                            + [f"modular_diffusion/content/{STORED}/result.json"], local_dir=f"{OUT}/hub")
SRC = f"{hub_dir}/modular_diffusion/content"
BLD = f"{hub_dir}/modular_diffusion/stages"
for t in BODIES:
    assert os.path.exists(f"{SRC}/{t}/result.json") and os.path.isdir(f"{SRC}/{t}/model_avg999"), f"the finished run {t} is not in the results repository"
for t in BUILDS:
    assert os.path.exists(f"{BLD}/{t}/result.json"), f"the build {t} is not in the results repository"
STORED_RESULT = json.load(open(f"{SRC}/{STORED}/result.json"))

SMOKE = f"{OUT}/smoke/plain"
smoke = content.main(variant="plain", seed=0, data=NPZ, out=SMOKE, device="cuda", scale=0.01, score=1, score_n=5000, ref=REF, reader=1)
assert "score" in smoke["score"], smoke["score"]
READER_CHECK = smoke["score"].get("reader_check")     # written by the job that computes the reference file: this one
print("class reader on the test pictures:", READER_CHECK, "| on the smoke model's samples:", (smoke["score"].get("class_read") or {}).get("accuracy"))
from torchmetrics.image.fid import FrechetInceptionDistance
bed = PicturesBed(NPZ, "cuda")
model = linker.link(f"{SMOKE}/model", device="cuda")
fake = score.sample_pictures(model, bed, 5000, steps=50, batch=1000, device="cuda")
real = bed.train_pictures()
fid = FrechetInceptionDistance(feature=2048, antialias=False).to("cuda")     # the same resize as the library's judge
for i in range(0, real.shape[0], 1000):
    fid.update(score.to_pictures(real[i:i + 1000], bed).to("cuda"), real=True)
for i in range(0, fake.shape[0], 1000):
    fid.update(fake[i:i + 1000].to("cuda"), real=False)
other = float(fid.compute())
mine = smoke["score"]["score"]
print(f"picture score: library {mine:.4f} | torchmetrics, the same resize {other:.4f} (relative difference {abs(mine - other) / other:.2e}) (5,000 pictures)")
del model, bed, real, fake, fid
torch.cuda.empty_cache()

# the new paths for a few steps each, with det=on as the job list runs them: a build on a frozen 4-block model and a
# grown one, the thaw pass at every level, experts born silent and by a solve (ten, and thirty by class and stage), a
# class added by a solve, members on averaged weights
S = f"{OUT}/smoke"
common = dict(seed=0, data=NPZ, device="cuda", scale=0.01, det="on", score=0)
time_split.main(variant="plain_d4", out=f"{S}/d4", avg="g8", **common)
stem = dict(stem_from=f"{S}/d4", stem_model="model_avg_g8")
stages.main(variant="tails", out=f"{S}/tails", total=6000, cuts="", **stem, **common)
stages.main(variant="grow", out=f"{S}/grow", blocks=2, steps_each=2000, **stem, **common)
sm_thaw = {k: stages.main(variant="thaw", out=f"{S}/thaw_{k}", base=f"{S}/tails", thaw=k, total=2000, **common) for k in ("tail", "1", "2", "all")}
sm_thaw["grown"] = stages.main(variant="thaw", out=f"{S}/thaw_grown", base=f"{S}/grow", thaw="all", total=2000, **common)
assert all(r["as_source"]["equals_stored"] and all(p["frozen_unchanged"] for p in r["parts"]) for r in sm_thaw.values())
for v in ("blind", "blind9"):
    content.main(variant=v, out=f"{S}/{v}", avg="g8", **common)
on = lambda v: dict(source=f"{S}/{v}", source_model="model_avg_g8")
sm_born = {v: content.main(variant=v, out=f"{S}/{v}", avg="g8", batches=5, **on("blind"), **common)
           for v in ("blind_experts10_id", "blind_experts10_solved", "blind_experts30_id", "blind_experts30_solved")}
sm_heads = closed_forms.main(variant="class_heads", out=f"{S}/class_heads", batches=5, **on("blind"), **common)
assert sm_born["blind_experts10_solved"]["at_birth"]["loss_over_levels"] == sm_heads["loss_over_levels"], "a solve-born model at birth is not the model of the solved heads"
assert all(r["exact"]["others_max_abs"] == 0.0 for r in sm_born.values())
sm_snap = content.main(variant="snap9_solved", out=f"{S}/snap9_solved", avg="g8", batches=5, **on("blind9"), **common)
sm_h9 = closed_forms.main(variant="class_stage_heads", out=f"{S}/snap9_heads3", batches=5, classes="9", only_classes="9", **on("blind9"), **common)
assert sm_snap["exact"]["others_max_abs"] == 0.0 and sm_h9["exact"]["others_max_abs"] == 0.0
sm_mem = {v: closed_forms.main(variant=v, out=f"{S}/{v}", steps=6000, source=SMOKE, **common) for v in ("members_avg", "member_all_avg")}
print("smoke of the new paths: thaw", {k: round(r["averaged"]["tail100"]["loss_over_levels"], 4) for k, r in sm_thaw.items()},
      "| born", {k: round(r["loss_over_levels"], 4) for k, r in sm_born.items()},
      "| members", {k: round(r["loss_over_levels"], 4) for k, r in sm_mem.items()})
INSTRUMENTS = {"reader_check": READER_CHECK, "smoke_class_read": smoke["score"].get("class_read"),
               "score_check": {"library": mine, "second_same_resize": other, "pictures": 5000}}

# the job list: (tag, experiment, arguments, the jobs it waits for, its cost in reference-model steps)
SCORE = 1700                                   # one scoring of 10,000 pictures, in reference-model steps
J = []
for s in (0, 1):
    tails, grown = f"base={BLD}/tails_k1_s{s}", f"base={BLD}/grow_1x3_s{s}"
    # the joint pass: the top part only (nothing thawed), with one or two blocks of the base, everything; 500 steps;
    # the grown build. Each is judged on the average of its last 100 steps, which is the one scored
    J += [(f"thaw_tail_s{s}", "stages", f"variant=thaw seed={s} {tails} thaw=tail total=2000 score_last=0", [], 1100 + SCORE),
          (f"thaw_1_s{s}", "stages", f"variant=thaw seed={s} {tails} thaw=1 total=2000 score_last=0", [], 1300 + SCORE),
          (f"thaw_2_s{s}", "stages", f"variant=thaw seed={s} {tails} thaw=2 total=2000 score_last=0", [], 1600 + SCORE),
          (f"thaw_all_s{s}", "stages", f"variant=thaw seed={s} {tails} thaw=all total=2000 score_last=0", [], 2000 + SCORE),
          (f"thaw_all_500_s{s}", "stages", f"variant=thaw seed={s} {tails} thaw=all total=500 score_last=0", [], 500 + SCORE),
          (f"thaw_all_grown_s{s}", "stages", f"variant=thaw seed={s} {grown} thaw=all total=2000 score_last=0", [], 2000 + SCORE)]
    # the reference trained straight through at the same compute as the build, and as the build with a full pass
    J += [(f"ref_{n}_s{s}", "stages", f"variant=source seed={s} total={n} avg=g8,tail100 score_last=0 score_avg=tail100", [], n + SCORE)
          for n in (6500, 8500)]
    # stage tails trained together with their stem, with the averaged weights kept
    J += [(f"tails_mixed_avg_s{s}", "time_split", f"variant=tails_mixed seed={s} avg=g8 score_last=0", [], 11000 + SCORE)]
for s in (0, 1):
    on_blind = f"seed={s} source={SRC}/blind_s{s} source_model=model_avg999 avg=g8 score_last=0"
    J += [(f"{v}_s{s}", "content", f"variant={v} {on_blind}", [], c + SCORE)
          for v, c in (("blind_experts10_id", 6500), ("blind_experts10_solved", 7000), ("blind_experts10_id_1500", 1700),
                       ("blind_experts10_solved_1500", 2200), ("blind_experts30_id", 7000), ("blind_experts30_solved", 7800))]
    on_blind9 = f"seed={s} source={SRC}/blind9_s{s} source_model=model_avg999"
    J += [(f"snap9_avg_s{s}", "content", f"variant=snap9 {on_blind9} avg=g8 score_last=0", [], 500 + SCORE),
          (f"snap9_solved_s{s}", "content", f"variant=snap9_solved {on_blind9} avg=g8 score_last=0", [], 1000 + SCORE),
          (f"snap9_head_s{s}", "closed_forms", f"variant=class_heads {on_blind9} batches=200 classes=9 only_classes=9", [], 500 + SCORE),
          (f"snap9_heads3_s{s}", "closed_forms", f"variant=class_stage_heads {on_blind9} batches=200 classes=9 only_classes=9", [], 700 + SCORE)]
    on_plain = f"seed={s} source={SRC}/plain_s{s} source_model=model_avg999 steps=6000 score_last=0"
    J += [(f"members3_plain_s{s}", "closed_forms", f"variant=members_avg {on_plain}", [], 9000 + SCORE),
          (f"member_all_plain_s{s}", "closed_forms", f"variant=member_all_avg {on_plain}", [], 4500 + SCORE)]
# a finished job of the batch before, run again: every gate now goes through the address frame, and its numbers must
# equal the stored ones to the last digit
J += [("frame_rerun_s0", "content", f"variant=blind_experts_all seed=0 source={SRC}/blind_s0 source_model=model_avg999 avg=g8 score_last=0", [], 6500 + SCORE)]
ALL = J
assert len({t for t, *_ in ALL}) == len(ALL), "a job tag is listed twice"
STEPS = {t: c + 900 for t, _, _, _, c in ALL}      # a job's time in reference-model steps (its training, its reads, its scoring)
RATE = 36.0                                        # the reference model's steps per second on this card type with det=on
total = sum(STEPS.values())
print(f"the job list: {len(ALL)} jobs, {total:,.0f} reference-model steps in all: about {total / RATE / 60:.0f} min")

One runner works the job list in order, every job with deterministic algorithms switched on (`det=on`), so a job repeats to the last digit. Each finished job folder is shipped to `REPO` under its experiment's own folder with the token from the notebook's secret `HF_TOKEN` (placed in this process's environment only; the jobs themselves run without it). The last job runs a finished job of the batch before once more: its numbers must equal the stored ones exactly. When the list has ended and its report is shipped, the machine is given back (`RELEASE`), since an idle machine keeps spending.

In [ ]:
# C3 - the runner, shipping after each job, a progress line a minute, the summary, the machine given back
from geolip.modular_diffusion import jobs, hub
PREFIX = {e: f"modular_diffusion/{e}" for e in ("content", "closed_forms", "stages", "time_split")}
RELEASE = True                              # give the machine back when the list has ended and the report is shipped
RUNS = f"{OUT}/runs"
EXTRA = ""                                   # more arguments for every job (empty in a real run)
kind = {}

def line(tag, exp, args, needs):
    kind[tag] = exp
    a = " " + args
    a += "" if " det=" in a else " det=on"
    a += "" if " device=" in a else " device=cuda threads=4"
    a += "" if " score=" in a else " score=1"
    a += " reader=1" if exp != "time_split" and " score=0" not in a else ""      # the class read needs a scored job
    return f"{tag} | 60 | geolip.modular_diffusion.run | {exp}{a} data={NPZ} ref={REF}{EXTRA}" + (f" | needs={','.join(needs)}" if needs else "")

tags = [t for t, *_ in ALL]
roster = f"{OUT}/roster.txt"
with open(roster, "w", encoding="utf-8", newline="\n") as fh:
    fh.write("\n".join(line(t, e, a, n) for t, e, a, n, _ in ALL) + "\nEND\n")
mark = lambda t, m: os.path.exists(f"{RUNS}/{t}/{m}")

if REPO:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    try:    # on a new machine: fetch this list's jobs that are already shipped and mark them done
        got_dir = snapshot_download(repo_id=REPO, allow_patterns=[f"{PREFIX[kind[t]]}/{t}/*" for t in tags], local_dir=f"{OUT}/hub_back")
        for t in tags:
            src = f"{got_dir}/{PREFIX[kind[t]]}/{t}"
            if os.path.exists(f"{src}/result.json") and not mark(t, "DONE"):
                shutil.copytree(src, f"{RUNS}/{t}", dirs_exist_ok=True)
                open(f"{RUNS}/{t}/DONE", "w").write("fetched back from the Hub\n")
    except Exception as e:
        print("nothing fetched back from the Hub:", repr(e)[:200])
for t in tags:      # a claim left by an interrupted session would hold its job forever
    if mark(t, "CLAIM") and not (mark(t, "DONE") or mark(t, "FAILED")):
        os.remove(f"{RUNS}/{t}/CLAIM")

def after_job(tag, folder, ok):
    if ok and REPO:
        hub.ship(folder, REPO, f"{PREFIX[kind[tag]]}/{tag}", f"{kind[tag].replace('_', ' ')}: {tag}")

t0 = time.time()
left0 = sum(STEPS[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
print(f"start: {sum(mark(t, 'DONE') for t in tags)}/{len(tags)} jobs already done; {left0:,.0f} steps to run")
runner = threading.Thread(target=jobs.run_roster, daemon=True,
                          kwargs=dict(roster=roster, out=RUNS, stop=f"{OUT}/STOP", log=f"{OUT}/roster.log", card_room_mib=None,
                                      ram_free_gb=4.0, max_hours=20.0, claim=True, after_job=after_job,
                                      env={"OMP_NUM_THREADS": "4", "MKL_NUM_THREADS": "4"}))
runner.start()
while runner.is_alive():
    runner.join(60)
    done, failed = sum(mark(t, "DONE") for t in tags), sum(mark(t, "FAILED") for t in tags)
    left = sum(STEPS[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
    spent = (time.time() - t0) / 60
    eta = spent * left / (left0 - left) if left0 > left else float("nan")
    print(f"jobs done {done}/{len(tags)} (failed {failed}); {spent:.0f} min spent; about {eta:.0f} min left", flush=True)

res = lambda t: json.load(open(f"{RUNS}/{t}/result.json")) if os.path.exists(f"{RUNS}/{t}/result.json") else None
again = res("frame_rerun_s0")
if again:
    pick = lambda r: {"loss_over_levels": r.get("loss_over_levels"), "average_loss": ((r.get("averaged") or {}).get("g8") or {}).get("loss_over_levels"),
                      "average_score": (((r.get("averaged") or {}).get("g8") or {}).get("score") or {}).get("score")}
    INSTRUMENTS["rerun"] = {"stored": pick(STORED_RESULT), "again": pick(again), "equal": pick(STORED_RESULT) == pick(again)}
    print("the stored job run again:", json.dumps(INSTRUMENTS["rerun"], indent=1))
summary = stages.summarize(RUNS) + "\n" + content.summarize(RUNS) + "\n" + closed_forms.summarize(RUNS) + "\n" + time_split.summarize(RUNS)
print(summary)
REPORT = f"{OUT}/report"
os.makedirs(REPORT, exist_ok=True)
open(f"{REPORT}/summary.md", "w", encoding="utf-8").write(summary)
json.dump(INSTRUMENTS, open(f"{REPORT}/instruments.json", "w", encoding="utf-8"), indent=1)   # the smoke's checks and the stored job run again
for f in ("roster.txt", "roster.log"):
    if os.path.exists(f"{OUT}/{f}"):
        shutil.copy(f"{OUT}/{f}", f"{REPORT}/{f.replace('.log', '_log.txt')}")
for t in tags:      # the logs of the jobs that failed (a job's own log files are not shipped with its folder)
    if mark(t, "FAILED"):
        for f in ("run.log", "log.txt"):
            if os.path.exists(f"{RUNS}/{t}/{f}"):
                shutil.copy(f"{RUNS}/{t}/{f}", f"{REPORT}/failed_{t}_{f.replace('.', '_')}.txt")
shipped = hub.ship(REPORT, REPO, "modular_diffusion/stages/_report_thaw", "the joint pass, births and members: summary and roster logs") if REPO else None
if RELEASE and REPO and shipped:
    from google.colab import runtime
    runtime.unassign()